In [ ]:
from datasets import load_dataset
import torch
from torch.utils.data import DataLoader
from transformers import BertTokenizer,BertConfig, BertForSequenceClassification
from torch.optim import AdamW
import tqdm

# Загрузка данных

In [6]:
imdb = load_dataset("imdb")
# Выберем 1000 примеров для обучения и 500 для теста
train_dataset = imdb['train'].shuffle(seed=42).select(range(1000))
test_dataset  = imdb['test'].shuffle(seed=42).select(range(500))

# Вывод первых 3 рецензий и меток
for i in range(3):
    print(f"Рецензия {i+1}: {train_dataset[i]['text']}")
    print(f"Метка {i+1}: {train_dataset[i]['label']}\n")

Рецензия 1: There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier's plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot similarities... The main character is weak and weirdo, but have "clairvoyance". People like to compare, to judge, to evaluate. How about just enjoying? Funny thing too, people writing Fortier looks American but, on the other hand, arguing they prefer American series (!!!). Maybe it's the language, or the spirit, but I think this series is more English than American. By the way, the actors are really good and funny. The acting is not superficial at all...
Метка 1: 1

Рецензия 2: This movie is a great. The plot is very true to the book which is a classic written by Mark Twain. The movie starts of with a scene where Hank sings a song with a bunch of kids called "when you stub your t

In [4]:
tokenizer = BertTokenizer.from_pretrained('google-bert/bert-base-uncased')

def tokenize_batch(batch):
    texts  = [x['text'] for x in batch]
    labels = [x['label'] for x in batch]
    encoding = tokenizer(
        texts,
        truncation=True,
        padding='max_length',
        max_length=256,
        return_tensors='pt'
    )
    return {
        'input_ids': encoding['input_ids'],
        'attention_mask': encoding['attention_mask'],
        'labels': torch.tensor(labels)
    }

def create_dataloader(dataset, batch_size=8, shuffle=True):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=tokenize_batch
    )

train_loader = create_dataloader(train_dataset)
test_loader  = create_dataloader(test_dataset, shuffle=False)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

# Модель

In [5]:
# Устройство: GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Конфигурация и модель
config = BertConfig.from_pretrained('google-bert/bert-base-uncased', num_labels=2)
model  = BertForSequenceClassification.from_pretrained(
    'google-bert/bert-base-uncased', config=config
).to(device)

# Оптимизатор
optimizer = AdamW(model.parameters(), lr=2e-5)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Обучение

In [7]:
from tqdm.auto import tqdm

epochs = 3
for epoch in range(1, epochs + 1):
    model.train()
    total_loss, total_correct, total_samples = 0, 0, 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch} [Train]"):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(
            input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss   += loss.item()
        preds         = outputs.logits.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

    train_loss = total_loss / len(train_loader)
    train_acc  = total_correct / total_samples

    model.eval()
    val_loss, val_correct, val_samples = 0, 0, 0

    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f"Epoch {epoch} [Eval]"):
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            outputs = model(
                input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            val_loss    += outputs.loss.item()
            preds        = outputs.logits.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_samples += labels.size(0)

    val_loss = val_loss / len(test_loader)
    val_acc  = val_correct / val_samples

    print(f"\nEpoch {epoch} results:")
    print(f"  Train: loss={train_loss:.4f}, acc={train_acc:.4f}")
    print(f"  Eval : loss={val_loss:.4f}, acc={val_acc:.4f}\n")

print(f"Final Eval Accuracy: {val_acc:.4f}")

Epoch 1 [Train]:   0%|          | 0/125 [00:00<?, ?it/s]

Epoch 1 [Eval]:   0%|          | 0/63 [00:00<?, ?it/s]


Epoch 1 results:
  Train: loss=0.4980, acc=0.7480
  Eval : loss=0.3302, acc=0.8780



Epoch 2 [Train]:   0%|          | 0/125 [00:00<?, ?it/s]

Epoch 2 [Eval]:   0%|          | 0/63 [00:00<?, ?it/s]


Epoch 2 results:
  Train: loss=0.2410, acc=0.9080
  Eval : loss=0.2824, acc=0.8800



Epoch 3 [Train]:   0%|          | 0/125 [00:00<?, ?it/s]

Epoch 3 [Eval]:   0%|          | 0/63 [00:00<?, ?it/s]


Epoch 3 results:
  Train: loss=0.0938, acc=0.9710
  Eval : loss=0.3594, acc=0.8920

Final Eval Accuracy: 0.8920
